# 02_matrix_mul / baseline 实验笔记本

对应仓库 `02_matrix_mul/baseline/mmul.cu`：最朴素的矩阵乘法实现，每个线程算输出矩阵 C 的一个元素，无任何优化（无 `restrict`、无共享内存/tiling）。

**第一步（手动操作，必须做）**：菜单栏 `代码执行程序` -> `更改运行时类型` -> 硬件加速器选 `T4 GPU` -> 保存。
然后从上到下依次运行下面每个代码块。

## 1. 检查 GPU 和 CUDA 是否就位

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 2. 拉取代码仓库
克隆整个仓库到 Colab 临时环境，包含 01~05 所有例子。

In [ ]:
!git clone https://github.com/Teneeduu/cuda_programming.git

## 3. 编译并运行 baseline，确认正确性
看到 `COMPLETED SUCCESSFULLY` 说明 GPU 算出来的结果和 CPU 校验一致。

源码里矩阵规模固定为 `N = 1 << 10`（即 1024x1024），每个线程块 `THREADS = 32`（即 32x32=1024 线程/块）。

In [ ]:
!nvcc -O3 -o mmul cuda_programming/02_matrix_mul/baseline/mmul.cu
!./mmul

## 4. 用 nsys 看一眼 kernel 实际耗时
重点看输出里 `matrixMul` kernel 的耗时，以及 `cudaMemcpy` 占比——baseline 版本里 kernel 本身因为访存效率差，耗时会比想象中长。

In [ ]:
!nsys profile --stats=true -o mmul_report ./mmul

## 5. 实验一：固定线程块大小，改变矩阵规模 N
baseline 是朴素的 O(N^3) 实现，且对矩阵 `b` 的访问是按列跨步（同一 warp 内访存地址跨度大，不合并），
理论上耗时会比单纯的 O(N^3) 增长更差。这里固定 `THREADS=32`，对比几组 N 的运行时间（`time` 包含了 CPU 端的结果校验，N 越大校验本身也会变慢，先用较小的几组）。

N 必须能被 THREADS（32）整除，下面几组 N 都满足。

In [ ]:
%%bash
cd cuda_programming/02_matrix_mul/baseline
for n in 256 512 1024; do
  cp mmul.cu mmul_exp.cu
  sed -i "s/int N = 1 << 10;/int N = ${n};/" mmul_exp.cu
  nvcc -O3 -o mmul_exp mmul_exp.cu
  echo "===== N=${n} ====="
  { time ./mmul_exp ; } 2>&1
  echo
done

## 6. 实验二：固定 N，改变线程块大小 THREADS
固定 `N=1024`，对比不同 `THREADS`（即每个 block 是 THREADS x THREADS 个线程）对耗时的影响。

注意：CUDA 单个 block 最多 1024 个线程，所以 `THREADS*THREADS <= 1024`，下面只能取 8 / 16 / 32。

In [ ]:
%%bash
cd cuda_programming/02_matrix_mul/baseline
for t in 8 16 32; do
  cp mmul.cu mmul_exp.cu
  sed -i "s/int THREADS = 32;/int THREADS = ${t};/" mmul_exp.cu
  nvcc -O3 -o mmul_exp mmul_exp.cu
  echo "===== THREADS=${t} ====="
  { time ./mmul_exp ; } 2>&1
  echo
done

## 7. 实验记录
把自己跑出来的数字和观察写在这里：N 增大时耗时是不是比 N^3 增长更快？THREADS 变小/变大对耗时有什么影响，和你的预期一致吗？

（这一格留空，自己填）